## tl;dr
La comparación con cinco folds temporales seleccionó **HistGradientBoosting**, con F1 macro medio 0,976 ± 0,007 y un IC bootstrap por mes de 0,972 a 0,980. En el test futuro de junio a agosto de 2026, obtuvo F1 macro 0,983 y accuracy 0,989. El conjunto de predicción nominal al 90 % cubrió 87,1 % del test, lo que muestra una caída de cobertura temporal que debe considerarse antes de usarlo.

El cuaderno combina los CSV fuente, imputa dentro de cada fold, estima incertidumbre y guarda el modelo y sus resultados.

## Contexto y método
La base contiene 21 500 registros, 24 reportes mensuales y 21 500 IDs distintos. Al no repetirse IDs, no corresponde usar GroupKFold. Como la fecha define reportes mensuales, se usa validación temporal expansiva: cinco folds de dos meses para seleccionar modelos en los primeros 18 meses, tres meses siguientes para calibración y los últimos tres meses como test futuro. La columna `atraso` queda fuera porque puede revelar directamente la etiqueta `mora`. La fecha y el ID tampoco entran como predictores. Se mantienen variables históricas (`mora_max_*`) bajo el supuesto que existen antes del pronóstico; hay que confirmar su disponibilidad temporal.

## 1. Cargar los datos originales
Suba los tres CSV originales `data_train-mora_final.csv`, `data_validation-mora_final.csv` y `data_test-mora_final.csv`. Se unirán para reconstruir el conjunto maestro antes de crear una separación temporal correcta.

In [ ]:
from google.colab import files  # Importa el selector de archivos de Google Colab.
archivos_cargados = files.upload()  # Solicita el archivo dataset_mora_completo_para_modelo_temporal.csv, sin imputar.

## 2. Importar herramientas

In [ ]:
import pandas as pd  # Importa pandas para leer y organizar los datos.
import numpy as np  # Importa NumPy para cálculos numéricos y remuestreo.
import matplotlib.pyplot as plt  # Importa Matplotlib para crear las gráficas.
import seaborn as sns  # Importa Seaborn para visualizar matrices de confusión.
import time  # Importa time para registrar duración del entrenamiento.
from IPython.display import display  # Importa display para mostrar tablas de forma legible en Colab.
from scipy.stats import t as distribucion_t  # Importa la distribución t de Student para intervalos de confianza.
from sklearn.compose import ColumnTransformer  # Permite transformar columnas numéricas y categóricas por separado.
from sklearn.pipeline import Pipeline  # Une imputación, transformación y estimador en un solo flujo.
from sklearn.impute import SimpleImputer  # Completa faltantes dentro del fold de entrenamiento.
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize  # Escala números, codifica categorías y binariza etiquetas.
from sklearn.linear_model import LogisticRegression  # Importa regresión logística como baseline.
from sklearn.tree import DecisionTreeClassifier  # Importa árbol de decisión.
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier  # Importa modelos de ensamble.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score, log_loss  # Importa métricas de clasificación y curvas.
from sklearn.utils.class_weight import compute_sample_weight  # Calcula pesos para compensar el desbalance de clases.
from sklearn.model_selection import TimeSeriesSplit  # Importa la validación cruzada temporal expansiva.
from sklearn.calibration import calibration_curve  # Importa la herramienta para analizar calibración de probabilidades.
from pathlib import Path  # Importa Path para crear carpetas de salida.
import joblib  # Importa joblib para guardar el modelo entrenado.

## 3. Reconstruir el conjunto maestro y auditar los meses

In [ ]:
ruta_datos = 'dataset_mora_completo_para_modelo_temporal.csv'  # Define el nombre del conjunto maestro que se debe cargar.
datos = pd.read_csv(ruta_datos, dtype={'ID': 'string'})  # Lee el CSV y conserva el identificador como texto.
datos['fecha_reporte'] = pd.to_datetime(datos['fecha_reporte'], errors='coerce')  # Convierte la fecha a tipo temporal.
datos = datos.sort_values('fecha_reporte').reset_index(drop=True)  # Ordena observaciones cronológicamente.
print('Filas y columnas:', datos.shape)  # Muestra el tamaño del conjunto.
print('Meses:', datos['fecha_reporte'].nunique())  # Cuenta los meses únicos.
print('IDs repetidos:', int(datos['ID'].duplicated().sum()))  # Verifica si existen clientes repetidos.
print('Duplicados exactos:', int(datos.duplicated().sum()))  # Verifica filas repetidas completas.
print('Faltantes totales:', int(datos.isna().sum().sum()))  # Cuenta faltantes antes de imputar.

## 4. Definir ventanas temporales sin mezclar futuro y pasado

In [ ]:
meses = np.array(sorted(datos['fecha_reporte'].dropna().unique()))  # Ordena los meses cronológicamente.
assert len(meses) == 24  # Comprueba que la fuente contiene los 24 reportes mensuales esperados.
meses_desarrollo = meses[:18]  # Reserva los primeros 18 meses para desarrollo y validación cruzada.
meses_calibracion = meses[18:21]  # Reserva los meses 19 a 21 para calibrar conjuntos de predicción.
meses_test = meses[21:]  # Reserva los últimos tres meses para la evaluación futura final.
print('Desarrollo:', meses_desarrollo[0], 'a', meses_desarrollo[-1])  # Imprime el periodo de desarrollo.
print('Calibración:', meses_calibracion[0], 'a', meses_calibracion[-1])  # Imprime el periodo de calibración.
print('Test futuro:', meses_test[0], 'a', meses_test[-1])  # Imprime el periodo final de test.

## 5. Revisar la distribución mensual de categorías

In [ ]:
carpeta_graficas = Path('resultados_validacion_temporal_mora')  # Define carpeta para exportar las figuras del análisis.
carpeta_graficas.mkdir(exist_ok=True)  # Crea la carpeta antes de guardar la primera gráfica.
distribucion = datos.groupby([datos['fecha_reporte'].dt.strftime('%Y-%m'), 'mora']).size().unstack(fill_value=0)  # Cuenta registros de cada clase en cada mes.
proporciones = distribucion.div(distribucion.sum(axis=1), axis=0)  # Convierte conteos mensuales a proporciones.
fig, ax = plt.subplots(figsize=(13, 6))  # Crea el lienzo para la composición mensual.
proporciones.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')  # Grafica proporciones mensuales por clase.
ax.set_title('Composición de categorías por mes de reporte')  # Coloca un título descriptivo.
ax.set_xlabel('Mes de reporte')  # Etiqueta el eje temporal.
ax.set_ylabel('Proporción de registros')  # Etiqueta el eje de proporciones.
ax.legend(title='Categoría de mora', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)  # Mueve la leyenda fuera del gráfico.
ax.tick_params(axis='x', rotation=45)  # Gira las etiquetas mensuales.
fig.tight_layout()  # Ajusta los márgenes.
fig.savefig(carpeta_graficas / 'distribucion_mensual_clases.png', dpi=170)  # Guarda la distribución mensual en la carpeta de resultados.
plt.show()  # Presenta la distribución mensual.

## 6. Preparar predictores y pipeline de imputación

In [ ]:
objetivo = 'mora'  # Define la categoría que se debe predecir.
columnas_excluidas = ['ID', 'fecha_reporte', 'atraso', objetivo]  # Excluye identificador, fecha y días de mora observados.
variables = [c for c in datos.columns if c not in columnas_excluidas]  # Selecciona predictores restantes.
numericas = [c for c in variables if pd.api.types.is_numeric_dtype(datos[c])]  # Identifica variables numéricas.
categoricas = [c for c in variables if c not in numericas]  # Identifica variables categóricas.
clases = sorted(datos[objetivo].astype(str).unique())  # Define el orden común de categorías.
def crear_pipeline(estimador):  # Define una función para aplicar las mismas transformaciones a cada modelo.
    transformador = ColumnTransformer([('numericas', Pipeline([('imputar', SimpleImputer(strategy='median')), ('escalar', StandardScaler())]), numericas), ('categoricas', Pipeline([('imputar', SimpleImputer(strategy='most_frequent')), ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categoricas)], remainder='drop', sparse_threshold=0)  # Imputa y transforma dentro del fold de entrenamiento.
    return Pipeline([('preprocesador', transformador), ('modelo', estimador)])  # Devuelve un pipeline completo y comparable.

## 7. Definir candidatos bajo las mismas condiciones

In [ ]:
candidatos = {  # Crea la colección de modelos que compartirán folds y predictores.
    'Regresión logística': LogisticRegression(max_iter=2500, class_weight='balanced', solver='lbfgs', C=1.0, random_state=42),  # Define el baseline interpretable.
    'Árbol de decisión': DecisionTreeClassifier(class_weight='balanced', max_depth=14, min_samples_leaf=8, random_state=42),  # Define un árbol regularizado.
    'Random Forest': RandomForestClassifier(n_estimators=250, class_weight='balanced_subsample', min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=42),  # Define bosque balanceado.
    'Extra Trees': ExtraTreesClassifier(n_estimators=250, class_weight='balanced', min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=42),  # Define árboles extra aleatorizados.
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=31, learning_rate=0.08, l2_regularization=1.0, early_stopping=False, random_state=42)  # Define boosting sin validación aleatoria interna.
}  # Finaliza el conjunto de algoritmos candidatos.

## 8. Ejecutar validación cruzada temporal expansiva
Cada fold valida dos meses consecutivos; el entrenamiento siempre usa meses anteriores. Los cinco modelos reciben exactamente las mismas ventanas.

In [ ]:
divisor_temporal = TimeSeriesSplit(n_splits=5, test_size=2)  # Define cinco folds expansivos con dos meses de validación cada uno.
registros_fold = []  # Crea la lista que almacenará métricas por modelo y fold.
predicciones_oof = {nombre: [] for nombre in candidatos}  # Crea listas para guardar predicciones fuera de fold.
for numero_fold, (indices_train, indices_val) in enumerate(divisor_temporal.split(meses_desarrollo), start=1):  # Recorre las ventanas temporales.
    fechas_train = meses_desarrollo[indices_train]  # Obtiene los meses disponibles para entrenar.
    fechas_val = meses_desarrollo[indices_val]  # Obtiene los meses futuros de validación.
    mascara_train = datos['fecha_reporte'].isin(fechas_train)  # Selecciona registros de meses de entrenamiento.
    mascara_val = datos['fecha_reporte'].isin(fechas_val)  # Selecciona registros de meses de validación.
    X_train = datos.loc[mascara_train, variables]  # Construye predictores del fold de entrenamiento.
    y_train = datos.loc[mascara_train, objetivo].astype(str)  # Construye etiquetas del fold de entrenamiento.
    X_val = datos.loc[mascara_val, variables]  # Construye predictores de la ventana futura de validación.
    y_val = datos.loc[mascara_val, objetivo].astype(str)  # Construye etiquetas de validación.
    pesos = compute_sample_weight(class_weight='balanced', y=y_train)  # Calcula pesos usando solo las etiquetas de entrenamiento.
    for nombre, estimador in candidatos.items():  # Ajusta todos los candidatos en la misma ventana.
        pipeline = crear_pipeline(estimador)  # Crea un pipeline nuevo para evitar compartir estado entre folds.
        inicio = time.perf_counter()  # Inicia el cronómetro del ajuste.
        if nombre == 'HistGradientBoosting':  # Comprueba si el estimador recibe pesos por fila.
            pipeline.fit(X_train, y_train, modelo__sample_weight=pesos)  # Ajusta el modelo con pesos calculados en train.
        else:  # Continúa para los modelos con ponderación interna.
            pipeline.fit(X_train, y_train)  # Ajusta el estimador con los datos históricos del fold.
        duracion = time.perf_counter() - inicio  # Calcula el tiempo de entrenamiento en segundos.
        pred = pipeline.predict(X_val)  # Predice los dos meses futuros de validación.
        prob = pipeline.predict_proba(X_val)  # Calcula probabilidades por categoría.
        registros_fold.append({'modelo': nombre, 'fold': numero_fold, 'inicio_validacion': str(pd.Timestamp(fechas_val[0]).date()), 'fin_validacion': str(pd.Timestamp(fechas_val[-1]).date()), 'filas_validacion': len(y_val), 'f1_macro': f1_score(y_val, pred, average='macro', zero_division=0), 'balanced_accuracy': balanced_accuracy_score(y_val, pred), 'accuracy': accuracy_score(y_val, pred), 'segundos_entrenamiento': duracion})  # Guarda métricas y ventana.
        salida_oof = pd.DataFrame({'fecha_reporte': datos.loc[mascara_val, 'fecha_reporte'].values, 'real': y_val.values, 'prediccion': pred})  # Crea tabla de predicciones fuera de fold.
        for indice_clase, clase in enumerate(pipeline.named_steps['modelo'].classes_):  # Recorre las clases en el orden del modelo.
            salida_oof[f'prob_{indice_clase}'] = prob[:, indice_clase]  # Guarda probabilidad predicha para cada clase.
        predicciones_oof[nombre].append(salida_oof)  # Conserva predicciones para intervalos y gráficas.

## 9. Resumir media, desviación e intervalos de confianza

In [ ]:
from scipy.stats import t as distribucion_t  # Importa t de Student para el intervalo de la media entre folds.
metricas_fold = pd.DataFrame(registros_fold)  # Convierte resultados por fold en una tabla.
filas_resumen = []  # Crea la lista para resumir desempeño y estabilidad.
generador = np.random.default_rng(42)  # Fija semilla para que el bootstrap se pueda reproducir.
repeticiones_bootstrap = 2000  # Define el número de remuestreos por bloques mensuales.
for nombre in candidatos:  # Calcula intervalos para cada algoritmo.
    resultados = metricas_fold[metricas_fold['modelo'] == nombre]  # Selecciona los folds del modelo actual.
    media = resultados['f1_macro'].mean()  # Calcula el F1 macro medio entre ventanas.
    desviacion = resultados['f1_macro'].std(ddof=1)  # Calcula la desviación entre ventanas temporales.
    error_estandar = desviacion / np.sqrt(len(resultados))  # Calcula el error estándar de la media.
    valor_t = distribucion_t.ppf(0.975, df=len(resultados) - 1)  # Obtiene el valor crítico t para 95 por ciento.
    pred_oof = pd.concat(predicciones_oof[nombre], ignore_index=True)  # Une predicciones generadas fuera de fold.
    bloques_mes = {mes: bloque for mes, bloque in pred_oof.groupby('fecha_reporte')}  # Agrupa filas por mes para remuestrear bloques completos.
    meses_oof = list(bloques_mes)  # Guarda meses disponibles para bootstrap.
    matrices_mes = np.stack([confusion_matrix(bloque['real'], bloque['prediccion'], labels=clases) for bloque in bloques_mes.values()])  # Resume cada mes con una matriz de confusión.
    valores_bootstrap = []  # Prepara resultados de bootstrap.
    for iteracion in range(repeticiones_bootstrap):  # Ejecuta remuestreos mensuales con reemplazo.
        indices_muestra = generador.integers(0, len(meses_oof), size=len(meses_oof))  # Selecciona índices de meses con reemplazo.
        matriz_muestra = matrices_mes[indices_muestra].sum(axis=0)  # Suma matrices de meses completos, conservando bloques temporales.
        verdaderos_positivos = np.diag(matriz_muestra)  # Extrae los aciertos de cada categoría.
        denominadores = matriz_muestra.sum(axis=0) + matriz_muestra.sum(axis=1)  # Calcula la suma de predicciones y observaciones de cada clase.
        f1_por_clase = np.divide(2 * verdaderos_positivos, denominadores, out=np.zeros_like(verdaderos_positivos, dtype=float), where=denominadores > 0)  # Calcula F1 por categoría.
        valores_bootstrap.append(float(f1_por_clase.mean()))  # Guarda el F1 macro de la réplica.
    filas_resumen.append({'modelo': nombre, 'f1_macro_media': media, 'f1_macro_desviacion': desviacion, 'f1_macro_ic95_t_inferior': max(0, media - valor_t * error_estandar), 'f1_macro_ic95_t_superior': min(1, media + valor_t * error_estandar), 'f1_macro_ic95_bootstrap_inferior': np.quantile(valores_bootstrap, 0.025), 'f1_macro_ic95_bootstrap_superior': np.quantile(valores_bootstrap, 0.975), 'balanced_accuracy_media': resultados['balanced_accuracy'].mean(), 'accuracy_media': resultados['accuracy'].mean(), 'criterio_conservador_media_menos_sd': media - desviacion, 'tiempo_medio_entrenamiento_seg': resultados['segundos_entrenamiento'].mean()})  # Registra promedio, incertidumbre, estabilidad y costo.
resumen = pd.DataFrame(filas_resumen).sort_values(['criterio_conservador_media_menos_sd', 'f1_macro_media'], ascending=False).reset_index(drop=True)  # Ordena primero por límite conservador de una desviación.
display(resumen)  # Muestra tabla comparativa de modelos.
mejor_nombre = resumen.loc[0, 'modelo']  # Selecciona modelo con mayor media menos una desviación estándar.
print('Modelo seleccionado por estabilidad y F1 macro:', mejor_nombre)  # Informa el modelo ganador.

## 10. Visualizar comparación e inestabilidad entre folds

In [ ]:
carpeta_graficas = Path('resultados_validacion_temporal_mora')  # Define la carpeta para guardar las figuras.
carpeta_graficas.mkdir(exist_ok=True)  # Crea la carpeta antes de exportar las gráficas.
fig, ax = plt.subplots(figsize=(11, 7))  # Crea la figura con intervalos bootstrap.
for posicion, fila in enumerate(resumen.sort_values('f1_macro_media').itertuples()):  # Recorre modelos ordenados por promedio.
    error_izq = fila.f1_macro_media - fila.f1_macro_ic95_bootstrap_inferior  # Calcula distancia hasta el límite inferior.
    error_der = fila.f1_macro_ic95_bootstrap_superior - fila.f1_macro_media  # Calcula distancia hasta el límite superior.
    color = '#e59a35' if fila.modelo == mejor_nombre else '#3973ac'  # Resalta el modelo ganador.
    ax.errorbar(fila.f1_macro_media, posicion, xerr=[[error_izq], [error_der]], fmt='o', capsize=4, color=color)  # Dibuja media e intervalo bootstrap.
orden = resumen.sort_values('f1_macro_media')['modelo']  # Recupera el orden de etiquetas del eje.
ax.set_yticks(range(len(orden)), orden)  # Coloca nombres de modelos en el eje vertical.
ax.set_xlim(0, 1)  # Define el rango de F1.
ax.set_xlabel('F1 macro fuera de fold con intervalo bootstrap por mes al 95 %')  # Explica métrica e intervalo.
ax.set_title('Comparación de desempeño e incertidumbre temporal')  # Añade título descriptivo.
ax.grid(axis='x', alpha=0.25)  # Añade una cuadrícula tenue para lectura.
fig.tight_layout()  # Ajusta márgenes.
fig.savefig(carpeta_graficas / 'comparacion_f1_ic_bootstrap.png', dpi=170)  # Guarda la comparación con incertidumbre.
plt.show()  # Presenta comparación de incertidumbre.
fig, ax = plt.subplots(figsize=(10, 6))  # Crea figura de estabilidad fold a fold.
for nombre, grupo in metricas_fold.groupby('modelo'):  # Recorre resultados por algoritmo.
    ax.plot(grupo['fold'], grupo['f1_macro'], marker='o', label=nombre)  # Dibuja cada modelo a través de los folds.
ax.set_xticks(sorted(metricas_fold['fold'].unique()))  # Define marcas para los cinco folds.
ax.set_ylim(0, 1.02)  # Fija escala común.
ax.set_xlabel('Fold temporal')  # Etiqueta los periodos.
ax.set_ylabel('F1 macro')  # Etiqueta la medida de desempeño.
ax.set_title('Estabilidad entre ventanas temporales')  # Titula el gráfico.
ax.legend(fontsize=8)  # Muestra la leyenda de modelos.
ax.grid(alpha=0.25)  # Añade cuadrícula ligera.
fig.tight_layout()  # Ajusta distribución.
fig.savefig(carpeta_graficas / 'estabilidad_por_fold.png', dpi=170)  # Guarda la variación entre folds.
plt.show()  # Presenta variabilidad por fold.
predicciones_ganador_oof = pd.concat(predicciones_oof[mejor_nombre], ignore_index=True)  # Une predicciones temporales fuera de fold del ganador.
matriz_oof = confusion_matrix(predicciones_ganador_oof['real'], predicciones_ganador_oof['prediccion'], labels=clases)  # Calcula confusión OOF del ganador.
fig, ax = plt.subplots(figsize=(10, 8))  # Crea figura para la confusión OOF.
sns.heatmap(matriz_oof, annot=True, fmt='d', cmap='Blues', xticklabels=clases, yticklabels=clases, ax=ax)  # Visualiza los aciertos y errores de los folds.
ax.set_title(f'Matriz de confusión OOF temporal — {mejor_nombre}')  # Titula la matriz fuera de fold.
ax.set_xlabel('Predicción')  # Etiqueta el eje de predicciones.
ax.set_ylabel('Clase real')  # Etiqueta el eje de clases observadas.
ax.tick_params(axis='x', rotation=35)  # Gira etiquetas para facilitar lectura.
fig.tight_layout()  # Ajusta márgenes de la matriz.
fig.savefig(carpeta_graficas / 'matriz_confusion_oof.png', dpi=170)  # Guarda la matriz temporal OOF.
plt.show()  # Presenta la matriz temporal OOF.

## 11. Entrenar ganador, calibrar y evaluar el test futuro

In [ ]:
mascara_train_final = datos['fecha_reporte'].isin(meses_desarrollo)  # Selecciona los primeros 18 meses.
mascara_calibracion = datos['fecha_reporte'].isin(meses_calibracion)  # Selecciona los meses de calibración posteriores.
mascara_test = datos['fecha_reporte'].isin(meses_test)  # Selecciona los tres meses finales.
X_train_final = datos.loc[mascara_train_final, variables]  # Crea predictores históricos finales.
y_train_final = datos.loc[mascara_train_final, objetivo].astype(str)  # Crea etiquetas para ajuste final.
X_calibracion = datos.loc[mascara_calibracion, variables]  # Crea predictores del periodo de calibración.
y_calibracion = datos.loc[mascara_calibracion, objetivo].astype(str)  # Crea etiquetas del periodo de calibración.
X_test = datos.loc[mascara_test, variables]  # Crea predictores del periodo futuro de test.
y_test = datos.loc[mascara_test, objetivo].astype(str)  # Crea etiquetas finales de test.
mejor_estimador = candidatos[mejor_nombre]  # Recupera configuración del ganador.
modelo_final = crear_pipeline(mejor_estimador)  # Crea pipeline final para el ganador.
pesos_finales = compute_sample_weight(class_weight='balanced', y=y_train_final)  # Calcula ponderaciones con entrenamiento final.
if mejor_nombre == 'HistGradientBoosting':  # Verifica si el candidato necesita pesos por fila.
    modelo_final.fit(X_train_final, y_train_final, modelo__sample_weight=pesos_finales)  # Ajusta boosting ponderado en meses de desarrollo.
else:  # Atiende los otros modelos balanceados internamente.
    modelo_final.fit(X_train_final, y_train_final)  # Ajusta el ganador en meses de desarrollo.
prob_cal = modelo_final.predict_proba(X_calibracion)  # Calcula probabilidades para el periodo de calibración.
pred_test = modelo_final.predict(X_test)  # Predice etiquetas del test futuro.
prob_test = modelo_final.predict_proba(X_test)  # Calcula probabilidades para el test futuro.
orden_clases = list(modelo_final.named_steps['modelo'].classes_)  # Obtiene el orden de clases del modelo final.
print(classification_report(y_test, pred_test, labels=orden_clases, zero_division=0))  # Presenta métricas por categoría en test.

## 12. Conjuntos predictivos para clasificación y cobertura observada
La presentación habla de intervalos para pronósticos numéricos. Como aquí la salida es una categoría, se calculan conjuntos predictivos conformes: cada registro puede recibir una o varias clases posibles. El umbral se calibra en meses anteriores y se verifica la cobertura en test. La cobertura temporal observada no garantiza cobertura futura si la distribución cambia.

In [ ]:
alpha = 0.10  # Define cobertura objetivo nominal del 90 por ciento.
indices_cal = np.array([orden_clases.index(etiqueta) for etiqueta in y_calibracion])  # Busca columna de probabilidad de la clase real.
puntajes_calibracion = 1 - prob_cal[np.arange(len(y_calibracion)), indices_cal]  # Calcula no conformidad como uno menos probabilidad real.
n_cal = len(puntajes_calibracion)  # Cuenta observaciones de calibración.
nivel_cuantil = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1)  # Calcula el cuantil conformal finito.
umbral = np.quantile(puntajes_calibracion, nivel_cuantil, method='higher')  # Fija el umbral con la calibración anterior al test.
conjuntos_calibracion = [[orden_clases[j] for j, prob in enumerate(fila) if 1 - prob <= umbral] for fila in prob_cal]  # Crea conjuntos sobre el periodo de calibración.
cubiertos_calibracion = np.array([real in conjunto for real, conjunto in zip(y_calibracion, conjuntos_calibracion)])  # Verifica cobertura de calibración.
conjuntos = [[orden_clases[j] for j, prob in enumerate(fila) if 1 - prob <= umbral] for fila in prob_test]  # Incluye clases con no conformidad bajo el umbral.
cubiertos = np.array([real in conjunto for real, conjunto in zip(y_test, conjuntos)])  # Comprueba si la clase real aparece en cada conjunto.
tamanos = np.array([len(conjunto) for conjunto in conjuntos])  # Calcula cuántas clases contiene cada conjunto.
print('Cobertura objetivo:', 1 - alpha)  # Muestra la cobertura nominal solicitada.
print('Cobertura observada en calibración:', cubiertos_calibracion.mean())  # Muestra cobertura retrospectiva del umbral calibrado.
print('Cobertura observada en test futuro:', cubiertos.mean())  # Muestra fracción de clases reales cubiertas en los meses futuros.
print('Conjuntos vacíos en test:', np.mean(tamanos == 0))  # Informa registros donde el conjunto conformal no retuvo una clase.
print('Tamaño medio de conjunto:', tamanos.mean())  # Muestra número medio de clases candidatas por registro.
print('Porcentaje de conjuntos unitarios:', (tamanos == 1).mean())  # Muestra proporción con una sola clase candidata.

## 13. Métricas, calibración de probabilidades y gráficas finales

In [ ]:
carpeta_graficas = Path('resultados_validacion_temporal_mora')  # Define la carpeta que guardará las gráficas.
carpeta_graficas.mkdir(exist_ok=True)  # Crea la carpeta antes de exportar figuras.
conjuntos_calibracion = [[orden_clases[j] for j, prob in enumerate(fila) if 1 - prob <= umbral] for fila in prob_cal]  # Construye conjuntos para las filas de calibración.
cubiertos_calibracion = np.array([real in conjunto for real, conjunto in zip(y_calibracion, conjuntos_calibracion)])  # Verifica cobertura en calibración.
metricas_test = {'accuracy': accuracy_score(y_test, pred_test), 'balanced_accuracy': balanced_accuracy_score(y_test, pred_test), 'precision_macro': precision_score(y_test, pred_test, average='macro', zero_division=0), 'recall_macro': recall_score(y_test, pred_test, average='macro', zero_division=0), 'f1_macro': f1_score(y_test, pred_test, average='macro', zero_division=0), 'f1_weighted': f1_score(y_test, pred_test, average='weighted', zero_division=0), 'log_loss': log_loss(y_test, prob_test, labels=orden_clases), 'cobertura_objetivo_90': 1-alpha, 'cobertura_observada_calibracion': cubiertos_calibracion.mean(), 'cobertura_observada_test': cubiertos.mean(), 'tamano_medio_conjunto_test': tamanos.mean()}  # Reúne métricas de clasificación y cobertura.
display(pd.DataFrame([metricas_test]).T.rename(columns={0: 'resultado'}))  # Muestra las medidas finales de test.
reporte_test = classification_report(y_test, pred_test, labels=orden_clases, output_dict=True, zero_division=0)  # Calcula métricas por cada clase.
fig, ejes = plt.subplots(1, 2, figsize=(18, 7))  # Crea paneles para confusión y F1 por clase.
matriz_test = confusion_matrix(y_test, pred_test, labels=orden_clases)  # Calcula conteos de predicciones por clase.
sns.heatmap(matriz_test, annot=True, fmt='d', cmap='Greens', xticklabels=orden_clases, yticklabels=orden_clases, ax=ejes[0])  # Dibuja la matriz de confusión.
ejes[0].set_title('Matriz de confusión del test futuro')  # Titula el panel de confusión.
ejes[0].set_xlabel('Predicción')  # Etiqueta el eje horizontal.
ejes[0].set_ylabel('Clase real')  # Etiqueta el eje vertical.
ejes[0].tick_params(axis='x', rotation=35)  # Gira etiquetas para que se lean.
ejes[1].barh(orden_clases, [reporte_test[c]['f1-score'] for c in orden_clases], color='#5583a8')  # Grafica F1 por categoría.
ejes[1].set_title('F1 por categoría en test')  # Titula el panel de F1.
ejes[1].set_xlabel('F1')  # Etiqueta el eje de F1.
ejes[1].set_xlim(0, 1.05)  # Define escala comparable entre categorías.
fig.tight_layout()  # Ajusta los dos paneles.
fig.savefig(carpeta_graficas / 'matriz_y_f1_por_clase_test.png', dpi=170)  # Guarda la figura de confusión y F1.
plt.show()  # Presenta matriz y F1 en Colab.
fig, ax = plt.subplots(figsize=(8, 5))  # Prepara comparación de cobertura conformal.
valores_cobertura = [1-alpha, cubiertos_calibracion.mean(), cubiertos.mean()]  # Reúne cobertura nominal, calibración y test.
ax.bar(['Objetivo 90 %', 'Calibración', 'Test futuro'], valores_cobertura, color=['#3973ac', '#7299b7', '#e59a35'])  # Grafica las tres coberturas.
ax.set_ylim(0, 1.05)  # Fija la escala de cobertura.
ax.set_ylabel('Fracción de casos cubiertos')  # Etiqueta el eje vertical.
ax.set_title('Cobertura observada de conjuntos predictivos')  # Titula el gráfico.
for indice, valor in enumerate(valores_cobertura):  # Recorre las tres barras.
    ax.text(indice, valor + 0.02, f'{valor:.1%}', ha='center')  # Anota el porcentaje sobre cada barra.
fig.tight_layout()  # Ajusta márgenes.
fig.savefig(carpeta_graficas / 'cobertura_conformal_calibracion_test.png', dpi=170)  # Guarda la figura de cobertura.
plt.show()  # Presenta la cobertura observada.
y_binario = label_binarize(y_test, classes=orden_clases)  # Convierte las etiquetas test a formato uno contra resto.
fig, ax = plt.subplots(figsize=(9, 7))  # Prepara gráfica ROC.
for i, clase in enumerate(orden_clases):  # Recorre categorías.
    falsos, verdaderos, _ = roc_curve(y_binario[:, i], prob_test[:, i])  # Calcula puntos ROC para una clase.
    ax.plot(falsos, verdaderos, label=f'{clase} AUC={auc(falsos, verdaderos):.3f}')  # Dibuja curva y AUC.
ax.plot([0, 1], [0, 1], 'k--', label='Azar')  # Incluye línea de referencia aleatoria.
ax.set_title('Curvas ROC uno contra resto en test futuro')  # Titula la curva.
ax.set_xlabel('Tasa de falsos positivos')  # Etiqueta el eje horizontal.
ax.set_ylabel('Tasa de verdaderos positivos')  # Etiqueta el eje vertical.
ax.legend(fontsize=7)  # Muestra categorías con sus AUC.
fig.tight_layout()  # Ajusta los elementos.
fig.savefig(carpeta_graficas / 'curvas_roc_test.png', dpi=170)  # Guarda la figura ROC.
plt.show()  # Muestra curvas ROC.
fig, ax = plt.subplots(figsize=(9, 7))  # Prepara gráfica precisión-recobrado.
for i, clase in enumerate(orden_clases):  # Recorre categorías.
    precision, recobrado, _ = precision_recall_curve(y_binario[:, i], prob_test[:, i])  # Calcula puntos PR.
    promedio_precision = average_precision_score(y_binario[:, i], prob_test[:, i])  # Calcula promedio de precisión.
    ax.plot(recobrado, precision, label=f'{clase} AP={promedio_precision:.3f}')  # Dibuja curva por clase.
ax.set_title('Precisión-recobrado en test futuro')  # Titula la figura.
ax.set_xlabel('Recobrado')  # Etiqueta eje horizontal.
ax.set_ylabel('Precisión')  # Etiqueta eje vertical.
ax.legend(fontsize=7)  # Muestra las categorías.
fig.tight_layout()  # Ajusta márgenes.
fig.savefig(carpeta_graficas / 'curvas_precision_recobrado_test.png', dpi=170)  # Guarda la figura PR.
plt.show()  # Presenta curvas PR.
fig, ax = plt.subplots(figsize=(9, 7))  # Prepara gráfico de calibración.
for i, clase in enumerate(orden_clases):  # Recorre categorías.
    frecuencia, prob_media = calibration_curve(y_binario[:, i], prob_test[:, i], n_bins=10, strategy='quantile')  # Resume frecuencia observada frente a probabilidad.
    ax.plot(prob_media, frecuencia, marker='o', label=clase)  # Dibuja curva de calibración de cada clase.
ax.plot([0, 1], [0, 1], 'k--', label='Calibración ideal')  # Añade línea de probabilidades bien calibradas.
ax.set_title('Calibración de probabilidades por clase en test')  # Titula el gráfico.
ax.set_xlabel('Probabilidad media predicha')  # Etiqueta eje X.
ax.set_ylabel('Frecuencia observada')  # Etiqueta eje Y.
ax.legend(fontsize=7)  # Muestra leyenda por clase.
ax.grid(alpha=0.2)  # Añade cuadrícula tenue.
fig.tight_layout()  # Ajusta espacio.
fig.savefig(carpeta_graficas / 'calibracion_probabilidades_test.png', dpi=170)  # Guarda la gráfica de calibración.
plt.show()  # Muestra calibración.


## 14. Guardar resultados, modelo y predicciones

In [ ]:
carpeta_salida = Path('resultados_validacion_temporal_mora')  # Define la carpeta de salida del análisis.
carpeta_salida.mkdir(exist_ok=True)  # Crea la carpeta si todavía no existe.
resumen.to_csv(carpeta_salida / 'comparacion_modelos_validacion_temporal.csv', index=False, encoding='utf-8-sig')  # Guarda ranking e intervalos de validación cruzada.
metricas_fold.to_csv(carpeta_salida / 'metricas_por_fold.csv', index=False, encoding='utf-8-sig')  # Guarda detalle por fold temporal.
pd.DataFrame([metricas_test]).to_csv(carpeta_salida / 'resultado_final_test_temporal.csv', index=False, encoding='utf-8-sig')  # Guarda resultado del periodo futuro.
pd.DataFrame(classification_report(y_test, pred_test, labels=orden_clases, output_dict=True, zero_division=0)).T.to_csv(carpeta_salida / 'reporte_por_clase_test.csv', encoding='utf-8-sig')  # Guarda métricas de cada clase.
predicciones = pd.DataFrame({'fecha_reporte': datos.loc[mascara_test, 'fecha_reporte'].values, 'real': y_test.values, 'prediccion': pred_test, 'probabilidad_maxima': prob_test.max(axis=1), 'conjunto_prediccion': [', '.join(s) for s in conjuntos], 'cubierto': cubiertos})  # Construye archivo auditable de predicciones.
for j, clase in enumerate(orden_clases):  # Recorre categorías para añadir probabilidades.
    predicciones[f'probabilidad_{j}'] = prob_test[:, j]  # Añade probabilidad estimada de cada clase.
predicciones.to_csv(carpeta_salida / 'predicciones_test_conjuntos.csv', index=False, encoding='utf-8-sig')  # Exporta predicciones por fila.
joblib.dump({'pipeline': modelo_final, 'nombre_modelo': mejor_nombre, 'variables': variables, 'clases': orden_clases, 'umbral_conformal_90': umbral}, carpeta_salida / 'modelo_temporal_ganador.joblib')  # Guarda el pipeline y sus metadatos.
print('Resultados guardados en:', carpeta_salida.resolve())  # Informa la ubicación de los resultados.
import shutil  # Importa shutil para crear un archivo ZIP de salida.
archivo_comprimido = shutil.make_archive('resultados_validacion_temporal_mora', 'zip', carpeta_salida)  # Comprime métricas, modelo, predicciones y figuras.
files.download(archivo_comprimido)  # Descarga el paquete comprimido desde Google Colab.

## Conclusiones y límites
La selección se basa en el F1 macro fuera de muestra temporal, penalizado por su desviación entre folds (media menos una desviación estándar). HistGradientBoosting obtiene 0,976 ± 0,007; sus intervalos se solapan con Random Forest, por lo que la diferencia no demuestra superioridad estadística. El intervalo t resume la variabilidad entre cinco folds; el bootstrap remuestrea meses completos para conservar estructura temporal. Ambos intervalos son aproximados porque las ventanas se solapan en entrenamiento y solo hay 24 meses. El conjunto de predicción conformal cubrió 90,1 % en calibración y 87,1 % en el test futuro; dejó el conjunto vacío en 12,9 % de los casos. Esta caída muestra que la cobertura no se mantuvo en el periodo futuro y exige revisar la calibración antes de cualquier uso. Confirme que variables `mora_max_*` sean conocidas antes de la predicción. Las etiquetas y proporciones mensuales parecen altamente regulares; los resultados no prueban desempeño en cartera bancaria real si la base es sintética.